# Stockout Analysis


## Project Overview

This notebook analyses 1 800 daily product-level records across 50 SKUs to measure stockout frequency, quantify lost sales, and identify seasonal and structural patterns.


## Learning Objectives

- Calculate stockout frequency by category and SKU
- Estimate lost sales in units and revenue
- Discover seasonal stockout patterns
- Explore the relationship between replenishment lead days and stockout rate


## Business or Research Problem

Stockouts result in lost revenue, customer dissatisfaction, and potential permanent loss of buyers. Identifying root causes and high-risk periods is essential for supply chain planning.


## Why This Analysis Matters

Even a 1% stockout rate across a product range can translate to millions in lost annual revenue. Proactive analysis enables better replenishment planning and safety stock sizing.


## Dataset Overview

- **Rows:** 1 800 daily product-level records
- **SKUs:** 50 products
- **Columns:** date, sku, category, opening_stock, demand, stockout_flag, lost_sales_units, replenishment_lead_days


## Dataset Source and License Notes

Synthetic dataset generated with NumPy seed=42. For educational purposes only.


## Environment Setup


In [ ]:
import sys, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
import warnings; warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
print('Python:', sys.version)


## Imports


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats


## Configuration / Constants


In [ ]:
SEED = 42
N = 1800
N_SKUS = 50
CATEGORIES = ['Electronics', 'Apparel', 'Food & Bev', 'Home Goods', 'Sports']
AVG_UNIT_PRICE = 45.0


## Dataset Download or Loading


In [ ]:
rng = np.random.default_rng(SEED)
skus = [f'SKU{str(i).zfill(3)}' for i in range(1, N_SKUS + 1)]
sku_arr = rng.choice(skus, size=N)
sku_cat = {s: rng.choice(CATEGORIES) for s in skus}
category = [sku_cat[s] for s in sku_arr]

dates = pd.date_range('2023-01-01', periods=N, freq='D')
date_arr = rng.choice(dates, size=N, replace=False)
date_arr = sorted(date_arr)

opening_stock = rng.integers(0, 200, N)
demand = rng.integers(5, 60, N)
stockout_flag = (opening_stock < demand).astype(int)
lost_sales_units = np.where(stockout_flag == 1, demand - opening_stock, 0)
replenishment_lead_days = rng.integers(1, 14, N)

df = pd.DataFrame({'date': date_arr, 'sku': sku_arr, 'category': category,
    'opening_stock': opening_stock, 'demand': demand,
    'stockout_flag': stockout_flag, 'lost_sales_units': lost_sales_units,
    'replenishment_lead_days': replenishment_lead_days})
df['month'] = pd.DatetimeIndex(df['date']).month
df['quarter'] = pd.DatetimeIndex(df['date']).quarter
df['estimated_lost_revenue'] = df['lost_sales_units'] * AVG_UNIT_PRICE
print(df.shape)
df.head(3)


## Data Validation Checks


In [ ]:
print('Missing values:\n', df.isnull().sum())
print('\nStockout rate:', df['stockout_flag'].mean().round(4))
assert df['lost_sales_units'].min() >= 0
print('Validation passed.')


## Data Cleaning


In [ ]:
df['opening_stock'] = df['opening_stock'].clip(lower=0)
print('Clean shape:', df.shape)


## Exploratory Data Analysis


In [ ]:
overall_stockout_rate = df['stockout_flag'].mean()
total_lost_units = df['lost_sales_units'].sum()
total_lost_rev = df['estimated_lost_revenue'].sum()
print(f'Overall stockout rate: {overall_stockout_rate:.2%}')
print(f'Total lost sales units: {total_lost_units:,}')
print(f'Estimated lost revenue: ${total_lost_rev:,.0f}')
print(df[['opening_stock','demand','lost_sales_units']].describe().round(2))


## Univariate Analysis


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df['opening_stock'], bins=20, color='steelblue', edgecolor='white')
axes[0].set_title('Opening Stock Distribution')
axes[0].set_xlabel('Units')
axes[1].hist(df['demand'], bins=20, color='coral', edgecolor='white')
axes[1].set_title('Daily Demand Distribution')
axes[1].set_xlabel('Units')
plt.tight_layout()
plt.show()


Opening stock is uniformly distributed while demand clusters in the mid-range, creating predictable stockout zones.


## Bivariate / Multivariate Analysis

Stockout frequency by category.


In [ ]:
cat_stockout = df.groupby('category').agg(
    stockout_rate=('stockout_flag', 'mean'),
    total_lost_units=('lost_sales_units', 'sum'),
    total_lost_rev=('estimated_lost_revenue', 'sum')
).sort_values('stockout_rate', ascending=False).reset_index()
print(cat_stockout.to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(cat_stockout['category'], cat_stockout['stockout_rate'] * 100, color='salmon')
ax.set_title('Stockout Rate by Category')
ax.set_ylabel('Stockout Rate (%)')
plt.tight_layout()
plt.show()


## Feature-Specific Insights

Top 10 SKUs by lost sales units.


In [ ]:
sku_lost = df.groupby('sku')['lost_sales_units'].sum().sort_values(ascending=False).head(10)
fig, ax = plt.subplots(figsize=(10, 4))
sku_lost.plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Top 10 SKUs by Lost Sales Units')
ax.set_ylabel('Lost Sales Units')
plt.tight_layout()
plt.show()
print(f'Top SKU by lost sales: {sku_lost.index[0]} with {sku_lost.iloc[0]} units lost')


## Statistical Checks

Correlation between replenishment lead days and stockout flag.


In [ ]:
corr, p_val = stats.pointbiserialr(df['replenishment_lead_days'], df['stockout_flag'])
print(f'Point-biserial correlation (lead days vs stockout): r={corr:.3f}, p={p_val:.4f}')
if p_val < 0.05:
    print('Significant: longer lead days correlate with higher stockout risk.')
else:
    print('No significant correlation detected.')


## Visual Analysis

Seasonal stockout pattern by month.


In [ ]:
monthly = df.groupby('month')['stockout_flag'].mean() * 100
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(monthly.index, monthly.values, marker='o', color='teal')
ax.set_title('Monthly Stockout Rate (%)')
ax.set_xlabel('Month')
ax.set_ylabel('Stockout Rate (%)')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
plt.tight_layout()
plt.show()
peak_month = monthly.idxmax()
print(f'Peak stockout month: Month {peak_month} at {monthly[peak_month]:.1f}%')


## Key Findings


In [ ]:
worst_cat = cat_stockout.iloc[0]
print(f'1. Overall stockout rate: {overall_stockout_rate:.2%}')
print(f'2. Total estimated lost revenue: ${total_lost_rev:,.0f}')
print(f'3. Worst category: {worst_cat["category"]} ({worst_cat["stockout_rate"]:.2%} stockout rate)')
print(f'4. Peak stockout month: {peak_month}')
print(f'5. Lead time-stockout correlation: r={corr:.3f}')


## Limitations

- Synthetic demand is uniform; real demand follows seasonal and promotional patterns
- Lost revenue estimate uses a flat average price; actual margin impact varies
- Replenishment is not modelled — stock never increases in this dataset


## Recommendations / Next Steps

1. Prioritise safety stock increase for top-lost-sales SKUs
2. Reduce lead days for highest-stockout categories via supplier renegotiation
3. Implement demand forecasting to anticipate stockout-risk periods
4. Set stockout alerts when opening stock falls below demand threshold


## Common Mistakes

- Measuring stockout rate without weighting by demand volume understates high-velocity SKU risk
- Using average unit price for lost revenue ignores margin differences
- Treating all stockouts equally without segmenting by priority tier


## Mini Challenge / Exercises

1. Compute stockout rate by SKU and identify the top 5 chronic stockout items
2. Build a scatter plot of `replenishment_lead_days` vs `stockout_flag` grouped by category
3. Calculate the weekly stockout rate and visualise as a time-series trend


## Final Summary / Key Takeaways

- Stockout analysis quantifies the hidden cost of poor inventory planning
- Category and SKU-level granularity reveals where intervention is most impactful
- Lead time reduction is a direct lever for reducing stockout probability
- Seasonal patterns guide pre-emptive safety stock adjustments
